# Ligand-Pocket QGNN: Quantum vs Classical Comparison

This notebook compares **Quantum** and **Classical** versions of the Ligand-Pocket QGNN model on the binding classification task.

**Architecture:**
- **Ligand**: Graph Neural Network (GCN) → Latent Vector
- **Pocket**: MLP → Latent Vector
- **Interaction**: Quantum Circuit (VQC) or Classical MLP → Probability

**Task:** Binary Classification (Binding vs Non-Binding)

In [ ]:
%load_ext autoreload
%autoreload 2

# ========== M4 CPU OPTIMIZATION - SET ENV VARS FIRST! ==========
import os
import sys

# CRITICAL: Set environment variables BEFORE importing torch or numpy
os.environ['OMP_NUM_THREADS'] = '12'
os.environ['MKL_NUM_THREADS'] = '12'
os.environ['VECLIB_MAXIMUM_THREADS'] = '12'
os.environ['NUMEXPR_NUM_THREADS'] = '12'

# Now import the rest
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json
import time
from datetime import datetime
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import multiprocessing as mp

# Set PyTorch threads (after import, before any operations)
torch.set_num_threads(12)
try:
    torch.set_num_interop_threads(12)
except RuntimeError:
    pass  # Already set

print("="*70)
print("⚡ M4 CPU OPTIMIZATION ENABLED")
print("="*70)
print(f"PyTorch threads: {torch.get_num_threads()}")
print(f"OMP threads: {os.environ['OMP_NUM_THREADS']}")
print("="*70)

# FIX FOR MACOS: Set multiprocessing start method to 'fork' for better stability
if sys.platform == 'darwin':
    try:
        mp.set_start_method('fork', force=True)
    except RuntimeError:
        pass  # Already set

from ligand_pocket_qgnn.data import LigandPocketDataProcessor, LigandPocketDataset
from ligand_pocket_qgnn.model import LigandPocketQGNN

# ========== OPTIMIZED COLLATE FUNCTION ==========
def optimized_collate_fn(batch):
    """
    Faster collate function using vectorized PyTorch operations.
    Replaces Python loops with batched tensor operations.
    """
    # Unzip batch
    x_list, edge_index_list, pocket_list, label_list = zip(*batch)

    # Fast concatenation
    pocket_batch = torch.stack(pocket_list)
    label_batch = torch.stack(label_list)

    # Calculate offsets vectorized
    num_nodes_list = torch.tensor([x.shape[0] for x in x_list], dtype=torch.long)
    cumsum = torch.cat([torch.zeros(1, dtype=torch.long), num_nodes_list.cumsum(0)])

    # Batch graphs
    x_batch = torch.cat(x_list, dim=0)

    # Shift edge indices vectorized
    edge_index_shifted = []
    for i, edge_index in enumerate(edge_index_list):
        if edge_index.shape[1] > 0:
            edge_index_shifted.append(edge_index + cumsum[i])

    if edge_index_shifted:
        edge_index_batch = torch.cat(edge_index_shifted, dim=1)
    else:
        edge_index_batch = torch.zeros((2, 0), dtype=torch.long)

    # Create batch vector (which sample each node belongs to)
    batch_vec = torch.cat([torch.full((n,), i, dtype=torch.long)
                           for i, n in enumerate(num_nodes_list)])

    return x_batch, edge_index_batch, batch_vec, pocket_batch, label_batch

print("✓ Imports and optimization loaded!")

## Configuration

In [2]:
# Paths
DATA_DIR = "/Users/priyanshudey/Code/Qunatum copy/othercode/data"
SAVE_DIR = "./ligand_pocket_comparison_results"
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. Load Data

In [3]:

# Data parameters
MAX_SAMPLES = 0  # Set to None for full dataset

SEED = 42069
SIMULATION_SEED = 42069  # For quantum simulation
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # OPTIMIZATION: Enable benchmark mode for faster training
    torch.backends.cudnn.deterministic = False  # Set to False for speed
    torch.backends.cudnn.benchmark = True  # Set to True for speed
# Initialize Processor
processor = LigandPocketDataProcessor(DATA_DIR, seed=SEED)

# Load Data
processor.load_data(max_samples=MAX_SAMPLES)

interactions = processor.get_dataset()
print(f"Total Interactions: {len(interactions)}")# Reproducibility

Searching for data in: /Users/priyanshudey/Code/Qunatum copy/othercode/data
Found 15239 protein descriptor files
Found 15239 protein descriptor files


Loading Data: 100%|██████████| 15239/15239 [00:45<00:00, 336.84it/s]


Generating negative samples (target: 122130)...


Generating Negatives:  16%|█▋        | 20043/122130 [00:00<00:00, 200423.98it/s]

⚠ Generated 9997/122130 negatives in batch. Filling remainder...


Generating Negatives: 100%|██████████| 122130/122130 [00:00<00:00, 131914.63it/s]


Loaded 13201 pockets, 122130 ligands
Interactions: 122130 positive, 122130 negative
Total Interactions: 244260


In [4]:
# # Hardware
# print(f"Using device: {DEVICE}")
# CPU_COUNT = mp.cpu_count()

# Model parameters - MUST-RUN EXPERIMENT CONFIGURATION
# ⚡ CRITICAL OPTIMIZATION: Reduce quantum circuit overhead
HIDDEN_DIM = 64
N_QUBITS = 6        # ⚡ REDUCED from 8 → 6 qubits (33% faster circuits)
N_QLAYERS = 2       # ⚡ REDUCED from 4 → 2 layers (50% faster)
QUANTUM_DEVICE = 'lightning.gpu'  # Try GPU first, falls back automatically

# # Training parameters - OPTIMIZED FOR CPU BOTTLENECK
# #BATCH_SIZE = int((GPU_MEMORY_GB * GPU_UTIL_TARGET) * SAMPLES_PER_GB / 12) * 128      # ⭐ DOUBLED: 512 → 1024 (reduce batch frequency)
# NUM_WORKERS = max(4, CPU_COUNT -3)       # ⭐ INCREASED: 8 → 12 (more parallel loading)
PIN_MEMORY = True
# PREFETCH_FACTOR = 4    # ⭐ NEW: Prefetch 4 batches per worker
EPOCHS = 100
LEARNING_RATE_QUANTUM = 0.001
LEARNING_RATE_CLASSICAL = 0.001
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15

print(f"\n{'='*70}")
print(f"⚡ QUANTUM CIRCUIT OPTIMIZATION")
print(f"{'='*70}")
print(f"Qubits: 8 → 6 (33% overhead reduction)")
print(f"Layers: 4 → 2 (50% overhead reduction)")
print(f"Expected speedup: 3-10x faster quantum evaluation")
print(f"Device: lightning.gpu (with lightning.qubit fallback)")
print(f"{'='*70}\n")

# print(f"\nMUST-RUN EXPERIMENT CONFIGURATION ")
# print(f"{'='*50}")
# print(f"  Random Seed: {SEED}")
# print(f"  Simulation Seed: {SIMULATION_SEED}")
# print(f"  Max Samples: {MAX_SAMPLES}")
# print(f"  Batch Size: {BATCH_SIZE} (INCREASED for GPU)")
# print(f"  Num Workers: {NUM_WORKERS} (INCREASED for CPU)")
# print(f"  Prefetch Factor: {PREFETCH_FACTOR}")
# print(f"  Epochs: {EPOCHS}")
# print(f"  Hidden Dim: {HIDDEN_DIM}")
# print(f"")
# print(f"  Qubits: {N_QUBITS}")
# print(f"  Quantum Layers (Depth): {N_QLAYERS}")
# print(f"  Quantum Device: {QUANTUM_DEVICE}")
# print(f"  Ansatz: StronglyEntanglingLayers")
# print(f"{'='*50}")


⚡ QUANTUM CIRCUIT OPTIMIZATION
Qubits: 8 → 6 (33% overhead reduction)
Layers: 4 → 2 (50% overhead reduction)
Expected speedup: 3-10x faster quantum evaluation
Device: lightning.gpu (with lightning.qubit fallback)



In [5]:
# ========== QUANTUM CIRCUIT PERFORMANCE ANALYSIS ==========
print(f"\n{'='*70}")
print(f"📊 EXPECTED PERFORMANCE WITH OPTIMIZATIONS")
print(f"{'='*70}")
print(f"""
Optimization Changes:
  ✓ Reduced qubits: 8 → 6 (fewer gates)
  ✓ Reduced layers: 4 → 2 (shallower circuit)
  ✓ Batch processing: Enabled in TorchLayer
  ✓ GPU acceleration: lightning.gpu for fast evaluation
  ✓ Gradient method: parameter-shift (parallelizable)

Expected Speedup:
  Before: ~1-2ms per sample with 8Q/4L
  After:  ~0.1-0.3ms per sample with 6Q/2L
  Speedup: 5-10x faster forward pass
""")
print(f"{'='*70}\n")


📊 EXPECTED PERFORMANCE WITH OPTIMIZATIONS

Optimization Changes:
  ✓ Reduced qubits: 8 → 6 (fewer gates)
  ✓ Reduced layers: 4 → 2 (shallower circuit)
  ✓ Batch processing: Enabled in TorchLayer
  ✓ GPU acceleration: lightning.gpu for fast evaluation
  ✓ Gradient method: parameter-shift (parallelizable)

Expected Speedup:
  Before: ~1-2ms per sample with 8Q/4L
  After:  ~0.1-0.3ms per sample with 6Q/2L
  Speedup: 5-10x faster forward pass




In [ ]:
# Create Datasets and Loaders - OPTIMIZED FOR QUANTUM SIMULATION
train_ints, val_ints = train_test_split(interactions, test_size=VAL_SPLIT, random_state=SEED)

train_dataset = LigandPocketDataset(processor, train_ints)
val_dataset = LigandPocketDataset(processor, val_ints)

# ⚡ OPTIMIZED FOR QUANTUM SIMULATION (not data loading!)
# The bottleneck is quantum circuit evaluation, NOT data loading
# Reduce workers to leave cores free for quantum simulation
import sys
import multiprocessing as mp

IS_MACOS = sys.platform == 'darwin'
CPU_COUNT = mp.cpu_count()  # Define CPU_COUNT here!

if IS_MACOS:
    # ⚡ KEY INSIGHT: Quantum simulation in PennyLane is SEQUENTIAL
    # Each sample processes through quantum circuit one-by-one
    # So we need to optimize for that, not for data loading parallelism
    
    # Use fewer workers - leave cores for quantum simulation
    EFFECTIVE_WORKERS = 2  # Reduced from 12 - minimal data loading
    EFFECTIVE_PREFETCH = 4
    USE_PERSISTENT = True
    
    # Use smaller batch size for more frequent work
    BATCH_SIZE = 2048  # Reduced from 12288 - more batches = more frequent CPU work
    
    print(f"\n{'='*70}")
    print(f"⚡ OPTIMIZED FOR SEQUENTIAL QUANTUM SIMULATION")
    print(f"{'='*70}")
    print(f"Total CPU cores: {CPU_COUNT}")
    print(f"\nDataLoader Settings:")
    print(f"  ✓ Workers: {EFFECTIVE_WORKERS} (minimal - data loading is NOT the bottleneck)")
    print(f"  ✓ Prefetch factor: {EFFECTIVE_PREFETCH}")
    print(f"  ✓ Batch size: {BATCH_SIZE} (smaller = more batches = better CPU utilization)")
    print(f"\nQuantum Simulation:")
    print(f"  • PennyLane processes samples SEQUENTIALLY")
    print(f"  • Each sample uses ~1-2 cores for matrix operations")
    print(f"  • {CPU_COUNT - EFFECTIVE_WORKERS} cores available for quantum simulation")
    print(f"\nExpected Results:")
    print(f"  • CPU usage: 40-60% (sequential quantum bottleneck)")
    print(f"  • ~{len(train_dataset) // BATCH_SIZE} batches per epoch")
    print(f"  • This is NORMAL for quantum simulation!")
    print(f"{'='*70}\n")
else:
    EFFECTIVE_WORKERS = 2
    EFFECTIVE_PREFETCH = 4
    BATCH_SIZE = 2048
    USE_PERSISTENT = True

# ⚡ OPTIMIZED DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=optimized_collate_fn,  
    num_workers=EFFECTIVE_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True if EFFECTIVE_WORKERS > 0 else False,
    prefetch_factor=EFFECTIVE_PREFETCH,
    timeout=120,
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=optimized_collate_fn,  
    num_workers=EFFECTIVE_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True if EFFECTIVE_WORKERS > 0 else False,
    prefetch_factor=EFFECTIVE_PREFETCH,
    timeout=120,
)

print(f"✓ DataLoaders created with quantum-optimized settings")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Samples per batch: {BATCH_SIZE}")
print(f"  Data loading workers: {EFFECTIVE_WORKERS}")
print(f"  Free cores for quantum: {CPU_COUNT - EFFECTIVE_WORKERS}\n")

In [11]:
# Create Datasets and Loaders - MAXIMUM M4 CPU UTILIZATION
train_ints, val_ints = train_test_split(interactions, test_size=VAL_SPLIT, random_state=SEED)

train_dataset = LigandPocketDataset(processor, train_ints)
val_dataset = LigandPocketDataset(processor, val_ints)

# ⚡ AGGRESSIVE M4 OPTIMIZATION: Use ALL cores (both P-cores and E-cores)
# M4 unified memory architecture can handle all cores efficiently
import sys
IS_MACOS = sys.platform == 'darwin'

if IS_MACOS:
    # ⚡ NEW STRATEGY: Use ALL available cores for maximum throughput
    # M4 unified memory makes E-cores almost as fast as P-cores for data loading
    # The key is high prefetch to keep all workers busy
    
    # Calculate workers: Use 80-90% of total cores (leave some for OS/main thread)
    EFFECTIVE_WORKERS = max(CPU_COUNT, int(CPU_COUNT * 0.95))
    
    # Aggressive prefetch: More batches prefetched = no worker idle time
    EFFECTIVE_PREFETCH = 8  # 8 batches per worker (very aggressive)
    
    # Use persistent workers to avoid fork overhead
    USE_PERSISTENT = True
    
    # ⚡ CRITICAL: Larger batch size reduces number of batches (fewer synchronizations)
    # This is the M4 sweet spot - fewer, larger batches > many small batches
    BATCH_SIZE = max(256, int(BATCH_SIZE * 1.5))  # 50% larger batches
    
    print(f"\n{'='*70}")
    print(f"⚡ MAXIMUM M4 CPU UTILIZATION CONFIGURATION")
    print(f"{'='*70}")
    print(f"Total CPU cores: {CPU_COUNT}")
    print(f"  • P-cores: ~6 (high performance)")
    print(f"  • E-cores: ~4 (efficiency)")
    print(f"\nDataLoader Settings:")
    print(f"  ✓ Workers: {EFFECTIVE_WORKERS}/{CPU_COUNT} cores ({EFFECTIVE_WORKERS*100//CPU_COUNT}% utilization)")
    print(f"  ✓ Persistent workers: {USE_PERSISTENT} (no fork overhead)")
    print(f"  ✓ Prefetch factor: {EFFECTIVE_PREFETCH} batches/worker")
    print(f"  ✓ Total prefetched: {EFFECTIVE_WORKERS * EFFECTIVE_PREFETCH} batches")
    print(f"  ✓ Batch size: {BATCH_SIZE} (increased for efficiency)")
    print(f"\nExpected Results:")
    print(f"  • CPU usage: ~95%+ across all cores")
    print(f"  • No idle workers")
    print(f"  • Faster data loading pipeline")
    print(f"{'='*70}\n")
else:
    EFFECTIVE_WORKERS = NUM_WORKERS
    EFFECTIVE_PREFETCH = PREFETCH_FACTOR
    USE_PERSISTENT = True

# ⚡ MAXIMUM PERFORMANCE DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=optimized_collate_fn,  
    num_workers=EFFECTIVE_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True if EFFECTIVE_WORKERS > 0 else False,
    prefetch_factor=EFFECTIVE_PREFETCH,
    timeout=120,  # Longer timeout for heavily loaded workers
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=optimized_collate_fn,  
    num_workers=EFFECTIVE_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True if EFFECTIVE_WORKERS > 0 else False,
    prefetch_factor=EFFECTIVE_PREFETCH,
    timeout=120,
)

print(f"✓ DataLoaders created with MAXIMUM M4 optimization")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Samples per batch: {BATCH_SIZE}")
print(f"  Total worker threads: {EFFECTIVE_WORKERS}")
print(f"  Expected CPU load: 90-95% across all cores\n")


⚡ MAXIMUM M4 CPU UTILIZATION CONFIGURATION
Total CPU cores: 12
  • P-cores: ~6 (high performance)
  • E-cores: ~4 (efficiency)

DataLoader Settings:
  ✓ Workers: 12/12 cores (100% utilization)
  ✓ Persistent workers: True (no fork overhead)
  ✓ Prefetch factor: 8 batches/worker
  ✓ Total prefetched: 96 batches
  ✓ Batch size: 12288 (increased for efficiency)

Expected Results:
  • CPU usage: ~95%+ across all cores
  • No idle workers
  • Faster data loading pipeline

✓ DataLoaders created with MAXIMUM M4 optimization
  Train batches: 16
  Val batches: 4
  Samples per batch: 12288
  Total worker threads: 12
  Expected CPU load: 90-95% across all cores



In [12]:
# Determine input dimensions from a sample
sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]

ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Ligand Input Dim: {ligand_dim}")
print(f"Pocket Input Dim: {pocket_dim}")

Ligand Input Dim: 10
Pocket Input Dim: 19


## 3. Training Functions

In [ ]:
# ========== DEVICE-SPECIFIC OPTIMIZATION DETECTION ==========
import subprocess
import platform

# Detect device and platform
PLATFORM = platform.system()
IS_MAC = PLATFORM == 'Darwin'
IS_CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if IS_CUDA else 'cpu'

print(f"\n{'='*70}")
print(f"🔍 SYSTEM DETECTION & OPTIMIZATION")
print(f"{'='*70}")
print(f"Platform: {PLATFORM}")
print(f"Device: {DEVICE.upper()}")
print(f"Is Mac: {IS_MAC}")
print(f"Is CUDA: {IS_CUDA}")

# Get GPU/CPU info
if IS_CUDA:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(f"CPU Cores: {CPU_COUNT}")
    try:
        result = subprocess.run(['sysctl', '-n', 'hw.memsize'], 
                              capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            total_mem_gb = int(result.stdout.strip()) / (1024**3)
            print(f"RAM: {total_mem_gb:.2f} GB")
    except:
        pass

print(f"{'='*70}\n")

In [16]:
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    """Train one epoch with progress tracking."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    for batch_idx, (x_batch, edge_index_batch, batch_vec, pocket_batch, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        # Non-blocking transfer to GPU
        x_batch = x_batch.to(device, non_blocking=True)
        edge_index_batch = edge_index_batch.to(device, non_blocking=True)
        batch_vec = batch_vec.to(device, non_blocking=True)
        pocket_batch = pocket_batch.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        
        # Forward pass
        outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'time': f'{batch_time:.1f}s'})
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}


def evaluate(model, criterion, loader, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x_batch, edge_index_batch, batch_vec, pocket_batch, labels in tqdm(loader, desc='Validation', leave=False):
            # Non-blocking transfer
            x_batch = x_batch.to(device, non_blocking=True)
            edge_index_batch = edge_index_batch.to(device, non_blocking=True)
            batch_vec = batch_vec.to(device, non_blocking=True)
            pocket_batch = pocket_batch.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }


def train_model(model, train_loader, val_loader, learning_rate, model_name, device, resume_from_checkpoint=None):
    """Train with detailed progress tracking and optional resume functionality."""
    print(f"\n{'='*70}")
    print(f"Training {model_name.upper()} Model")
    print(f"{'='*70}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Batches per epoch: {len(train_loader)}\n")
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    criterion = nn.BCELoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [],
        'learning_rates': []
    }
    
    best_val_auc = 0.0
    patience_counter = 0
    start_epoch = 0
    
    # Resume from checkpoint if provided
    if resume_from_checkpoint is not None:
        print(f"{'='*70}")
        print(f"RESUMING FROM CHECKPOINT")
        print(f"{'='*70}")
        
        checkpoint_path = os.path.join(SAVE_DIR, f"{model_name}_best.pt")
        history_path = os.path.join(SAVE_DIR, f"{model_name}_history.json")
        
        if os.path.exists(checkpoint_path) and os.path.exists(history_path):
            # Load checkpoint
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            best_val_auc = checkpoint['best_auc']
            
            # Load history
            with open(history_path, 'r') as f:
                history = json.load(f)
            
            start_epoch = len(history['train_loss'])
            
            # Calculate patience counter (epochs since last improvement)
            best_epoch = np.argmax(history['val_auc'])
            patience_counter = start_epoch - 1 - best_epoch
            
            print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch'] + 1}")
            print(f"✓ Best AUC so far: {best_val_auc:.4f} (epoch {best_epoch + 1})")
            print(f"✓ Resuming from epoch {start_epoch + 1}")
            print(f"✓ Current patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            
            # Restore scheduler state by running it on past history
            for auc in history['val_auc']:
                scheduler.step(auc)
            
            print(f"✓ Current learning rate: {optimizer.param_groups[0]['lr']:.6f}")
            print(f"{'='*70}\n")
        else:
            print(f"⚠ Checkpoint files not found, starting from scratch")
            print(f"{'='*70}\n")
    
    start_time = datetime.now()
    
    for epoch in range(start_epoch, EPOCHS):
        epoch_start = datetime.now()
        print(f"\n{'='*70}")
        print(f"EPOCH {epoch+1}/{EPOCHS} - Started at {epoch_start.strftime('%H:%M:%S')}")
        print(f"{'='*70}")
        
        train_metrics = train_epoch(model, optimizer, criterion, train_loader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        
        # Update scheduler
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        
        if current_lr != old_lr:
            print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
        
        # Save metrics
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        history['learning_rates'].append(current_lr)
        
        epoch_time = (datetime.now() - epoch_start).total_seconds()
        
        # Print results
        print(f"\n{'-'*70}")
        print(f"Epoch {epoch+1} Results ({epoch_time:.1f}s):")
        print(f"  Train: loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}")
        print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
              f"auc={val_metrics['auc']:.4f}, f1={val_metrics['f1']:.4f}")
        print(f"  Best AUC so far: {best_val_auc:.4f}")
        print(f"{'-'*70}")
        
        # Save best model
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_val_auc,
                'history': history
            }, os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            
            print(f"✓ New best AUC: {best_val_auc:.4f} (saved)")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        
        # Save history every epoch
        with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
            json.dump(history, f, indent=2)
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n{model_name.upper()} Training Complete!")
    print(f"  Total time: {total_time/60:.2f} minutes")
    print(f"  Best AUC: {best_val_auc:.4f}")
    
    return history, best_val_auc

In [17]:
# ========== QUANTUM-AWARE TRAINING ==========
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    """Train one epoch with quantum circuit batch optimization."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    # ⚡ CRITICAL: Use torch.cuda.stream() for GPU/Quantum overlap
    if device == 'cuda':
        stream = torch.cuda.Stream()
    
    for batch_idx, (x_batch, edge_index_batch, batch_vec, pocket_batch, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        # ⚡ Overlap GPU compute with next batch preparation
        if device == 'cuda':
            with torch.cuda.stream(stream):
                x_batch = x_batch.to(device, non_blocking=True)
                edge_index_batch = edge_index_batch.to(device, non_blocking=True)
                batch_vec = batch_vec.to(device, non_blocking=True)
                pocket_batch = pocket_batch.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
            torch.cuda.current_stream().wait_stream(stream)
        else:
            x_batch = x_batch.to(device, non_blocking=True)
            edge_index_batch = edge_index_batch.to(device, non_blocking=True)
            batch_vec = batch_vec.to(device, non_blocking=True)
            pocket_batch = pocket_batch.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Forward pass
        outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'time': f'{batch_time:.1f}s'})
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}


# ...existing code...

## 4. Train Quantum Model

In [18]:
print("Creating Quantum Model...")
quantum_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=True,
    quantum_device=QUANTUM_DEVICE  # Pass the quantum device
)

print(f"\n{'='*70}")
print(f"QUANTUM MODEL ARCHITECTURE")
print(f"{'='*70}")
print(quantum_model)
print(f"{'='*70}")
print(f"Quantum Simulation Seed: {SIMULATION_SEED}")
print(f"{'='*70}")

print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Resume from checkpoint - set to True to continue training
quantum_history, quantum_best_auc = train_model(
    quantum_model,
    train_loader,
    val_loader,
    LEARNING_RATE_QUANTUM,
    "quantum",
    DEVICE,
    resume_from_checkpoint=False  # ← RESUME FROM CHECKPOINT
)

print(f"\nQuantum training finished at: {datetime.now().strftime('%H:%M:%S')}")

Creating Quantum Model...
⚠ Warning: lightning.gpu not available (Device lightning.gpu does not exist. Make sure the required plugin is installed.)
  Falling back to lightning.qubit


AttributeError: module 'pennylane' has no attribute 'ExecutionConfig'

## 5. Train Classical Model

In [ ]:
print("Creating Classical Model...")
classical_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=False,  # Classical version
)

print(f"\nClassical Model Architecture:")
print(classical_model)

print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

classical_history, classical_best_auc = train_model(
    classical_model,
    train_loader,
    val_loader,
    LEARNING_RATE_CLASSICAL,
    "classical",
    DEVICE
)

print(f"\nClassical training finished at: {datetime.now().strftime('%H:%M:%S')}")

## 6. Comparison Results

In [ ]:
print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"Quantum Model     - Best AUC: {quantum_best_auc:.4f}")
print(f"Classical Model   - Best AUC: {classical_best_auc:.4f}")
print(f"\nQuantum Advantage: {(quantum_best_auc - classical_best_auc)*100:.2f}% {'improvement' if quantum_best_auc > classical_best_auc else 'deficit'}")
print("="*70)

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss
axes[0, 0].plot(quantum_history['train_loss'], label='Quantum Train', color='blue', alpha=0.7)
axes[0, 0].plot(classical_history['train_loss'], label='Classical Train', color='orange', alpha=0.7)
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(quantum_history['val_loss'], label='Quantum Val', color='blue', alpha=0.7)
axes[1, 0].plot(classical_history['val_loss'], label='Classical Val', color='orange', alpha=0.7)
axes[1, 0].set_title('Validation Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(quantum_history['train_acc'], label='Quantum Train', color='blue', alpha=0.7)
axes[0, 1].plot(classical_history['train_acc'], label='Classical Train', color='orange', alpha=0.7)
axes[0, 1].set_title('Training Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(quantum_history['val_acc'], label='Quantum Val', color='blue', alpha=0.7)
axes[1, 1].plot(classical_history['val_acc'], label='Classical Val', color='orange', alpha=0.7)
axes[1, 1].set_title('Validation Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# AUC
axes[0, 2].plot(quantum_history['val_auc'], label='Quantum', color='blue', marker='o', alpha=0.7)
axes[0, 2].plot(classical_history['val_auc'], label='Classical', color='orange', marker='s', alpha=0.7)
axes[0, 2].axhline(y=quantum_best_auc, color='blue', linestyle='--', alpha=0.5, label=f'Q Best: {quantum_best_auc:.4f}')
axes[0, 2].axhline(y=classical_best_auc, color='orange', linestyle='--', alpha=0.5, label=f'C Best: {classical_best_auc:.4f}')
axes[0, 2].set_title('Validation AUC')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('AUC')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# F1 Score
axes[1, 2].plot(quantum_history['val_f1'], label='Quantum', color='blue', alpha=0.7)
axes[1, 2].plot(classical_history['val_f1'], label='Classical', color='orange', alpha=0.7)
axes[1, 2].set_title('Validation F1 Score')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('F1')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'comparison_plots.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlots saved to {SAVE_DIR}/comparison_plots.png")

## 8. Final Metrics Summary

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Get best epoch metrics for both models
q_best_idx = np.argmax(quantum_history['val_auc'])
c_best_idx = np.argmax(classical_history['val_auc'])

results_df = pd.DataFrame({
    'Model': ['Quantum', 'Classical'],
    'Best Epoch': [q_best_idx + 1, c_best_idx + 1],
    'Val Loss': [
        quantum_history['val_loss'][q_best_idx],
        classical_history['val_loss'][c_best_idx]
    ],
    'Val Acc': [
        quantum_history['val_acc'][q_best_idx],
        classical_history['val_acc'][c_best_idx]
    ],
    'Val AUC': [quantum_best_auc, classical_best_auc],
    'Val Precision': [
        quantum_history['val_precision'][q_best_idx],
        classical_history['val_precision'][c_best_idx]
    ],
    'Val Recall': [
        quantum_history['val_recall'][q_best_idx],
        classical_history['val_recall'][c_best_idx]
    ],
    'Val F1': [
        quantum_history['val_f1'][q_best_idx],
        classical_history['val_f1'][c_best_idx]
    ]
})

print("\n" + "="*70)
print("SUMMARY OF BEST METRICS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Get final predictions for detailed classification report
quantum_model.eval()
classical_model.eval()

with torch.no_grad():
    # Quantum predictions
    q_all_preds, q_all_labels = [], []
    for x, edge_idx, pocket_feat, label in val_loader:
        x, edge_idx, pocket_feat = x.to(DEVICE), edge_idx.to(DEVICE), pocket_feat.to(DEVICE)
        output = quantum_model(x, edge_idx, pocket_feat)
        q_all_preds.extend(output.cpu().numpy())
        q_all_labels.extend(label.numpy())
    
    # Classical predictions
    c_all_preds, c_all_labels = [], []
    for x, edge_idx, pocket_feat, label in val_loader:
        x, edge_idx, pocket_feat = x.to(DEVICE), edge_idx.to(DEVICE), pocket_feat.to(DEVICE)
        output = classical_model(x, edge_idx, pocket_feat)
        c_all_preds.extend(output.cpu().numpy())
        c_all_labels.extend(label.numpy())

q_all_preds = np.array(q_all_preds).flatten()
q_all_labels = np.array(q_all_labels)
c_all_preds = np.array(c_all_preds).flatten()
c_all_labels = np.array(c_all_labels)

q_pred_binary = (q_all_preds > 0.5).astype(int)
c_pred_binary = (c_all_preds > 0.5).astype(int)

# Print detailed classification reports
print("\n" + "="*70)
print("QUANTUM MODEL - DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(q_all_labels, q_pred_binary, 
                          target_names=['Non-Binding', 'Binding'],
                          digits=4))

print("\n" + "="*70)
print("CLASSICAL MODEL - DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(c_all_labels, c_pred_binary, 
                          target_names=['Non-Binding', 'Binding'],
                          digits=4))

# Confusion matrices
q_cm = confusion_matrix(q_all_labels, q_pred_binary)
c_cm = confusion_matrix(c_all_labels, c_pred_binary)

print("\n" + "="*70)
print("CONFUSION MATRICES")
print("="*70)
print("\nQuantum Model:")
print(f"  True Negatives:  {q_cm[0,0]}")
print(f"  False Positives: {q_cm[0,1]}")
print(f"  False Negatives: {q_cm[1,0]}")
print(f"  True Positives:  {q_cm[1,1]}")

print("\nClassical Model:")
print(f"  True Negatives:  {c_cm[0,0]}")
print(f"  False Positives: {c_cm[0,1]}")
print(f"  False Negatives: {c_cm[1,0]}")
print(f"  True Positives:  {c_cm[1,1]}")

# Model comparison
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
q_acc = (q_cm[0,0] + q_cm[1,1]) / q_cm.sum()
c_acc = (c_cm[0,0] + c_cm[1,1]) / c_cm.sum()

comparison_data = {
    'Metric': ['Accuracy', 'AUC-ROC', 'Precision', 'Recall', 'F1-Score'],
    'Quantum': [
        f"{q_acc:.4f}",
        f"{quantum_best_auc:.4f}",
        f"{quantum_history['val_precision'][q_best_idx]:.4f}",
        f"{quantum_history['val_recall'][q_best_idx]:.4f}",
        f"{quantum_history['val_f1'][q_best_idx]:.4f}"
    ],
    'Classical': [
        f"{c_acc:.4f}",
        f"{classical_best_auc:.4f}",
        f"{classical_history['val_precision'][c_best_idx]:.4f}",
        f"{classical_history['val_recall'][c_best_idx]:.4f}",
        f"{classical_history['val_f1'][c_best_idx]::.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("="*70)

# Save results
results_df.to_csv(os.path.join(SAVE_DIR, 'comparison_results.csv'), index=False)
comparison_df.to_csv(os.path.join(SAVE_DIR, 'detailed_comparison.csv'), index=False)
print(f"\nResults saved to {SAVE_DIR}/comparison_results.csv")
print(f"Detailed comparison saved to {SAVE_DIR}/detailed_comparison.csv")

In [ ]:
# ========== PERFORMANCE PROFILING BEFORE TRAINING ==========
print(f"\n{'='*70}")
print(f"⚡ PERFORMANCE PROFILING (BENCHMARK)")
print(f"{'='*70}")

# Profile data loading speed
print(f"\nTesting data loader speed...")
data_load_times = []
for i, (x, edge_idx, batch_vec, pocket, label) in enumerate(train_loader):
    if i >= 5:  # Test first 5 batches
        break
    start = time.time()
    # Simulate GPU transfer
    x = x.to(DEVICE, non_blocking=True)
    edge_idx = edge_idx.to(DEVICE, non_blocking=True)
    batch_vec = batch_vec.to(device=DEVICE, non_blocking=True)
    pocket = pocket.to(DEVICE, non_blocking=True)
    label = label.to(DEVICE, non_blocking=True)
    torch.cuda.synchronize() if IS_CUDA else None
    elapsed = time.time() - start
    data_load_times.append(elapsed)
    print(f"  Batch {i+1}: {elapsed*1000:.2f}ms (size: {len(label)})")

avg_data_time = np.mean(data_load_times)
print(f"\n✓ Average data load time: {avg_data_time*1000:.2f}ms per batch")

# Profile quantum circuit speed (small test)
print(f"\nTesting quantum circuit speed...")
if N_QUBITS <= 4:
    # Create tiny test batch
    test_latent = torch.randn(1, HIDDEN_DIM, device=DEVICE)
    test_pocket = torch.randn(1, pocket_dim, device=DEVICE)
    
    # Time quantum forward pass
    quantum_times = []
    for trial in range(3):
        start = time.time()
        # Create a tiny quantum model for testing
        if IS_CUDA:
            # CUDA is fast - time multiple passes
            for _ in range(5):
                _ = quantum_model.interaction_layer(test_latent, test_pocket)
        else:
            # Mac might be slow - just one pass
            _ = quantum_model.interaction_layer(test_latent, test_pocket)
        torch.cuda.synchronize() if IS_CUDA else None
        elapsed = time.time() - start
        quantum_times.append(elapsed)
    
    avg_quantum_time = np.mean(quantum_times)
    print(f"  Average quantum circuit time: {avg_quantum_time*1000:.2f}ms")
    
    # Estimate per-batch quantum time
    batch_quantum_est = avg_quantum_time * (BATCH_SIZE / (5 if IS_CUDA else 1))
    print(f"  Estimated per-batch quantum time: {batch_quantum_est*1000:.2f}ms")
else:
    print(f"  Skipping quantum benchmark (circuit too large)")

print(f"{'='*70}\n")

# Detect bottleneck
print(f"\n{'='*70}")
print(f"🔍 BOTTLENECK ANALYSIS")
print(f"{'='*70}")
if IS_CUDA:
    print(f"\n⚠️  CUDA DETECTED - Quantum circuits are the bottleneck!")
    print(f"  → Reduce N_QUBITS and N_QLAYERS for faster training")
    print(f"  → Current: {N_QUBITS} qubits, {N_QLAYERS} layers")
    print(f"  → Recommended: 3-4 qubits, 1 layer")
else:
    print(f"\n💾 MAC DETECTED - Data loading is the bottleneck!")
    print(f"  → Maximize CPU workers for parallel data loading")
    print(f"  → Current: {EFFECTIVE_WORKERS} workers, {EFFECTIVE_PREFETCH} prefetch")
    print(f"  → Already optimized for M-series unified memory")

print(f"{'='*70}\n")